# Setting up PEST with PstFrom

In this notebook we wrap the DIZON model from `00_model` in a PEST interface with `pyemu`'s `PstFrom`. We add parameters for the uncertain aquifer properties, decision variables for the supply well, and observations and forecasts from the model outputs. At the end we draw the prior parameter ensemble that `02_priormc` will run.

In [ ]:
import os
import numpy as np
import pandas as pd
import flopy
import pyemu
import shutil
import herebedragons as hbd
import matplotlib.pyplot as plt
from pathlib import Path

# Getting Started

In [ ]:
# folder containing original model files
org_d = Path('model')

# a dir to hold a copy of the org model files
tmp_d = Path('tmp')

if Path.exists(tmp_d):
    shutil.rmtree(tmp_d)
shutil.copytree(org_d,tmp_d)

# get executables if 00_model not run
hbd.get_bins(tmp_d)

In [ ]:
# if you haven't run the first tutorial
# run the model once to make sure it works
# pyemu.os_utils.run("mf6rtm",cwd=tmp_d)

In [ ]:
# load simulation
sim = flopy.mf6.MFSimulation.load(sim_ws=tmp_d)
# load flow model
gwf = sim.get_model()

This model was built using a `DISV` grid. `PstFrom` is going to require the `flopy` model grid object to setup pilot points and spatially varying covariance:

In [ ]:
sr = gwf.modelgrid
sr

# Initializing PST FROM

In [ ]:
# specify a template directory (i.e. the PstFrom working folder)
template_ws = Path("pst_template")

# instantiate PstFrom
pf = pyemu.utils.PstFrom(original_d=tmp_d,       # where the model is stored
                            new_d=template_ws,   # the PEST template folder
                            remove_existing=True, # ensures a clean start
                            longnames=True,      # set False if using PEST/PEST_HP
                            spatial_reference=sr, #the spatial reference we generated earlier
                            zero_based=False, # does the MODEL use zero based indices? For example, MODFLOW does NOT
                            echo=False) # to stop PstFrom from writing lots of information to the notebook; experiment by setting it as True to see the difference; useful for troubleshooting

# Add Parameters

Even a flow-only model can carry a lot of parameters once we add hydraulic conductivity and storage. Transport and reactions add many more properties we could parameterize.

To keep this tutorial manageable we parameterize one flow property (hydraulic conductivity), one transport property (porosity) and one reaction property (the initial pyrite content). Then we add the pumping rates of the supply well as decision variables.

## Hydraulic conductivity

The model has no `idomain`, so we make a "zone array" of ones. `PstFrom` needs it to set up pilot points. It has one value per cell in a layer:

In [ ]:
ib = np.ones(sr.ncpl, dtype=int)
ib

In [ ]:
# we need to get the Hk files. For that we are gonna use a helper function that gets the filenames from a tag

tag = "npf_k_"
files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')

# flopy does not write nice and tidy array files so we are gonna do that here

for f in files:
    fpath = template_ws / f
    hbd.tidy_array(fpath)


In [ ]:
#Let's check one file against idomain
k = np.loadtxt(template_ws / files[0])

k.shape, ib.shape # all good!

In [ ]:
# exponential variogram for spatially varying parameters
v_pp = pyemu.geostats.SphVario(contribution=1.0, #sill
                                    a=200, # range of correlation; length units of the model. In our case 'meters'
                                    anisotropy=1, #name says it all
                                    bearing=0 #angle in degrees East of North corresponding to anisotropy ellipse
                                    )

# geostatistical structure for spatially varying parameters
pp_gs = pyemu.geostats.GeoStruct(variograms=v_pp, transform='log') 

pp_gs.plot()

In [ ]:
lb = 0.001
ub = 10.0
for f in files:
    tag = f.split('.')[1].replace("_",".")
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    # ult_ubound = 10,
                    # ult_lbound = 1e-5, # tiny
                    pp_options={"prep_hyperpars":False,
                                "pp_space":25} #in this case is meters
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="constant",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    # ult_ubound = 10,
                    # ult_lbound = 1e-5, # tiny
                    )

In [ ]:
df_pp.head()

In [ ]:
fig, ax = plt.subplots(1,1)
mv = flopy.plot.PlotMapView(gwf)
mv.plot_grid()
ax.scatter(df_pp.x, df_pp.y, s=10)

## Porosity

Transport properties such as porosity (`mst_porosity`) or longitudinal dispersivity (`dsp_alh`) are set separately for each transported species. MODFLOW 6 does this on purpose, so species can differ.

Here we want the same porosity for every species. We parameterize porosity for one species and copy its files to the others before each model run. A small function in the forward run does the copying.

In [ ]:
# let's get a list of species first.

species = sim.model_names[1:] # the flow model is always fist so we can skip that one

species

We parameterize H2O and copy its porosity files to the other species.

In [ ]:
tag = "h2o.mst_porosity_"
files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')

# flopy does not write nice and tidy array files so we are gonna do that here

for f in files:
    fpath = template_ws / f
    hbd.tidy_array(fpath)


lb = 0.5
ub = 1.5
for f in files:
    tag = f.split('.')[1].replace("_",".")
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 0.65, # max porosity is 1 but we don't wanna go over 0.6
                    ult_lbound = 5e-2, # tiny just in case
                    pp_options={"prep_hyperpars":False,
                                "pp_space":25} #in this case is meters
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="constant",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 0.65, # max poro is 1
                    ult_lbound = 0.05, # tiny
                    )

## Kinetic phases: pyrite

The last property is the initial pyrite content (`m0`) in each cell. The pyrite oxidation rate scales with the pyrite left in a cell, so these parameters also carry the uncertainty in the reaction rate.

In [ ]:
tag = "kinetic_phases.pyrite.m0"
files = hbd.get_input_filenames(tag, template_ws=template_ws, extension='.txt')

# mf6rtm writes these files as tidy array already.. Winning

lb = 0.05
ub = 5

for f in files:
    tag = f.split('.txt')[0].replace("_",".").lower()
    df_pp = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="pilotpoints",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 15, #
                    ult_lbound = 1e-5, # tiny just in case
                    pp_options={"prep_hyperpars":False,
                                "pp_space":25} #in this case is meters
                    )
    df_cn = pf.add_parameters(f,
                    zone_array=ib,
                    par_type="zone",
                    par_style="m",
                    geostruct=pp_gs,
                    par_name_base=tag,
                    pargp=tag,
                    lower_bound=lb,
                    upper_bound=ub,
                    ult_ubound = 10, #
                    ult_lbound = 1e-5, # tiny
                    )
    _ = pf.add_observations(f, prefix=tag, obsgp=tag)


## Decision variables

The supply well `wellopt` pumps from three screens (`welopt-ly1`, `welopt-ly3` and `welopt-ly5`) from stress period 21 on. We give each screen a multiplier on its design rate, between 0 (off) and 2 (twice the design rate). These are our **decision variables**: values we control, not properties of the aquifer. The optimization in `07_mou` will choose them.

One multiplier per screen applies to all stress periods, so we pass every `welopt` file at once and index the rows by the screen name (column 4, the boundname):

In [ ]:
files = hbd.get_input_filenames("welopt_stress_period_data", template_ws=template_ws)

df_dv = pf.add_parameters(files,
                    par_type="grid",
                    par_style="m",
                    index_cols=[4], # the screen name
                    use_cols=[2], # the pumping rate
                    par_name_base="welopt",
                    pargp="decvar",
                    lower_bound=0.0,
                    upper_bound=2.0,
                    transform="none", # a multiplier of 0 switches a screen off, so no log transform
                    )
df_dv.drop_duplicates("parnme")[["parnme", "parval1", "parlbnd", "parubnd"]]

# Pre- and post-processing functions

As we discussed earlier, we want to have some functions before and after we run the forward model. PyEMU is all about python, so we are going to do a forward_run.py script here using pyEMU. 

In [ ]:
pf.extra_py_imports.append("flopy")
pf.extra_py_imports.append("shutil")
pf.extra_py_imports.append("from pathlib import Path")

pf.add_py_function("herebedragons.py","tidy_array()",is_pre_cmd=None)
pf.add_py_function("herebedragons.py","get_input_filenames()",is_pre_cmd=None)
pf.add_py_function("herebedragons.py","extract_layer_number()",is_pre_cmd=None)


pf.add_py_function("herebedragons.py","copy_parameterized_transport_files()",is_pre_cmd=True)
pf.mod_sys_cmds.append("mf6rtm")

In [ ]:
hbd.copy_parameterized_transport_files(ws=template_ws)

In [ ]:
pf.write_forward_run()

In [ ]:
pst = pf.build_pst()


In [ ]:
pst.write(template_ws / 'pest.pst',version=2)

In [ ]:
pst.parameter_data

# Add Observations

In [ ]:
# process_sim_conc pairs the simulated concentrations with the measured ones, interpolating in time
pf.extra_py_imports.append("from flopy.utils.gridintersect import GridIntersect")
pf.add_py_function("herebedragons.py","node_to_layer_icell2d()",is_pre_cmd=None)
pf.add_py_function("herebedragons.py","time_interpolate()",is_pre_cmd=None)
pf.add_py_function("herebedragons.py","process_sim_conc()",is_pre_cmd=False)
pf.add_py_function("herebedragons.py","thin_conc_obs()",is_pre_cmd=False)

# run the functions once to create the files
hbd.process_sim_conc(wd=template_ws)
fname, df = hbd.thin_conc_obs(wd=template_ws)
print(len(pd.read_csv(template_ws / '_obs.conc.simvsmeas.csv')), 'simulated values,', len(df), 'kept')

The model writes concentrations every day, which is about 220,000 values. PEST does not need all of them. `thin_conc_obs` keeps the values at the field-sample times and one value a week for plotting, which is about 37,000.

In [ ]:
# add the observations
prefix = 'conc'
_ = pf.add_observations(fname,
                        index_cols=['time','obsid','variable'], # these columns identify each observation
                        use_cols='sim', # the values to track
                        prefix=prefix,
                        obsgp=prefix)

## Decision variables and forecasts as observations

The optimization in `07_mou` needs to see the decision variables in the model outputs too, so we record the pumped rate of each screen after every run.

We also add the **forecasts**: the quantities we care about for the decision. Ours is the peak sulfate in the water pumped by `wellopt` after stress period 21. We record it for each screen and for the blended water, which is the rate-weighted mean of the three screens. `process_welopt` writes both files:

In [ ]:
pf.add_py_function("herebedragons.py","process_welopt()",is_pre_cmd=False)
rates, fcs = hbd.process_welopt(wd=template_ws)

_ = pf.add_observations("_welopt.rates.csv",
                        index_cols="screen",
                        use_cols="q",
                        prefix="welopt",
                        obsgp="decvar_obs")
_ = pf.add_observations("_welopt.so4peak.csv",
                        index_cols="forecast",
                        use_cols="value",
                        prefix="fc",
                        obsgp="forecast")
fcs

In [ ]:
pst = pf.build_pst()

In [ ]:
# let's do some organization in the pst observation data for better filtering
obs = pst.observation_data
obs['oname'] = obs['variable'].copy() #tidy up

forecasts = obs.loc[obs.obgnme=="forecast", "obsnme"].tolist()
pst.pestpp_options["forecasts"] = forecasts

pst.write(template_ws/'pest.pst',version=2)
forecasts

# Draw Prior Ensemble

In [ ]:
# build the prior covariance matrix and store it as a compressed binary file (otherwise it can get huge!)
cov = pf.build_prior(fmt='coo', filename=Path(template_ws,"prior_cov.jcb"))
pf.pst.pestpp_options["parcov"] = "prior_cov.jcb"

# take a peek at the matrix; zeros are left blank
x = cov.x.copy()
x[x==0] = np.nan
plt.imshow(x)

Now we draw 1000 realisations from the prior. The decision variables have no spatial structure and we want the emulator in `03_dsi` to see the whole range of pumping rates, so we replace their draws with uniform ones between the bounds.

In [ ]:
pe = pf.draw(num_reals=1000, use_specsim=False) # draw parameters from the prior distribution
pe.enforce() # enforces parameter bounds

par = pst.parameter_data
dvs = par.loc[par.pargp=="decvar", "parnme"]
rng = np.random.default_rng(42)
pe.loc[:, dvs] = rng.uniform(par.loc[dvs, "parlbnd"], par.loc[dvs, "parubnd"], size=(pe.shape[0], len(dvs)))

pe.to_binary(Path(template_ws,"prior_pe.jcb")) #writes the parameter ensemble to binary file

pst.pestpp_options["ies_par_en"] = "prior_pe.jcb" #tells pestpp to use prior_pe
pst.pestpp_options["ies_num_reals"] = 1000

pst.write(Path(template_ws,"pest.pst"),version=2)

# Test run

Before we start a thousand runs, we run the model once through PEST with the starting parameter values (`noptmax` = 0). One run takes about 9 minutes. If everything is wired up, the simulated values match the observation values we just recorded, and phi is zero.

In [ ]:
pst.control_data.noptmax = 0
pst.write(Path(template_ws,"pest.pst"),version=2)
pyemu.os_utils.run('pestpp-ies pest.pst', cwd=template_ws)

In [ ]:
pst = pyemu.Pst(str(template_ws/"pest.pst"))
pst.phi < 1e-15 # sanity check

# Check Fields

Let's look at the K and porosity fields of one realisation. We write that realisation's parameter values into the model input files without running the model.

In [ ]:
pst.parameter_data.loc[:, "parval1"] = pe.loc[pe.index[40], pst.par_names].values
pst.write_input_files(pst_path=template_ws)
# apply the multipliers the same way the forward run does
cwd = Path.cwd()
os.chdir(template_ws)
pyemu.helpers.apply_list_and_array_pars(arr_par_file="mult2model_info.csv")
os.chdir(cwd)

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws = template_ws,
                                sim_name = 'gwf', 
                                version='mf6',
                                exe_name='mf6',
                                verbosity_level=0)
gwf = sim.get_model("gwf")
gwt = sim.get_model("h2o")

poro = gwt.mst.porosity.get_data()
k = gwf.npf.k.get_data()

plot_arrays = [k, poro]
plot_titles = ['H$_k$', 'Porosity']

fig, axs = plt.subplots(1,2, figsize=(12,3),
                        squeeze=True, sharex=True, sharey=True)
for ax, a, t in zip(axs, plot_arrays, plot_titles):
    mv = flopy.plot.PlotMapView(gwf, layer=6, ax=ax)
    mv.plot_grid()
    arr = mv.plot_array(a)
    cbar = plt.colorbar(arr, ax=ax, shrink=1)
    ax.set_title(t)
    ax.set_xlabel("X (m)")
    ax.set_ylabel("Y (m)")
fig.tight_layout()